Alerta de ordens pendentes, criação de KPI:
Ordens pendentes, quantidade de itens e informações de clientes para envio de alerta por email e telefone de quem tem cadastro completo.


In [0]:
%python
gold_path     = '/Volumes/bikestore/resource/gold/'


In [0]:
describe bikestore.logistics.silver_orders 

In [0]:
SELECT 
  o.customer_id
 ,sum(o.quantity) AS quantity
 ,o.store_name
 ,o.order_date
 --,upper(o.order_status) AS order_status
FROM  bikestore.logistics.silver_orders AS o
WHERE upper(o.order_status) = 'PENDING'
GROUP BY o.customer_id,o.store_name,o.order_date
LIMIT 10;

In [0]:
with order_pending as(
  SELECT 
  o.customer_id
 ,sum(o.quantity) AS quantity
 ,o.store_name
 ,o.order_date
 --,upper(o.order_status) AS order_status
FROM  bikestore.logistics.silver_orders AS o
WHERE upper(o.order_status) = 'PENDING'
GROUP BY o.customer_id,o.store_name,o.order_date
)
SELECT 
 op.*
 ,c.first_name AS first_name_customer
 ,c.email 
 ,c.phone  
FROM order_pending op
LEFT JOIN bikestore.logistics.silver_custumers c ON op.customer_id = c.customer_id
WHERE c.email IS NOT null
AND c.phone IS NOT null



In [0]:
%python
df_orders_pending = spark.sql("""
                              
    with order_pending as(
    SELECT 
        o.customer_id
        ,sum(o.quantity) AS quantity
        ,o.store_name
        ,o.order_date
    --,upper(o.order_status) AS order_status
    FROM  bikestore.logistics.silver_orders AS o
    WHERE upper(o.order_status) = 'PENDING'
    GROUP BY o.customer_id,o.store_name,o.order_date
    )
    SELECT 
        op.*
        ,c.first_name AS first_name_customer
        ,c.email 
        ,c.phone  
    FROM order_pending op
    LEFT JOIN bikestore.logistics.silver_custumers c ON op.customer_id = c.customer_id
    WHERE c.email IS NOT null
    AND c.phone IS NOT null


""")

df_orders_pending.write\
    .mode('overwrite')\
    .format('delta')\
    .option('mergeSchmema','true')\
    .save(f'{gold_path}/order_pending_gold')

In [0]:
CREATE OR REPLACE TABLE  bikestore.logistics.order_pending_gold AS(

  with order_pending as(
  SELECT 
      o.customer_id
      ,sum(o.quantity) AS quantity
      ,o.store_name
      ,o.order_date
  --,upper(o.order_status) AS order_status
  FROM  bikestore.logistics.silver_orders AS o
  WHERE upper(o.order_status) = 'PENDING'
  GROUP BY o.customer_id,o.store_name,o.order_date
  )
  SELECT 
      op.*
      ,c.first_name AS first_name_customer
      ,c.email 
      ,c.phone  
  FROM order_pending op
  LEFT JOIN bikestore.logistics.silver_custumers c ON op.customer_id = c.customer_id
  WHERE c.email IS NOT null
  AND c.phone IS NOT null
)